# 02 Cleaning

| Issue | Rule |
|---|---|
| Sheet overlap (1-9 Dec 2010, verified identical) | Drop the second sheet's copy |
| Negative qty, not a "C" invoice, no customer ID (stock write-offs) | Drop |
| Price <= 0 (incl. "Adjust bad debt") | Drop |
| Non-product stock codes (postage, fees, discounts, manual, test, gift vouchers, DOT, B) | Drop, via explicit list |
| Exact repeated lines within an invoice (about 0.3% of revenue) | Keep |
| Missing customer ID | Keep for overall revenue, exclude from customer-level work |
| Cancellations ("C" invoices) | Keep, flagged; netted against purchases later in SQL |

In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/online_retail_raw.csv",
                 parse_dates=["invoice_date"],
                 dtype={"invoice": str, "stock_code": str})
df["revenue"] = df["quantity"] * df["price"]
print(df.shape)   # expect (1067371, 10)

(1067371, 10)


In [2]:
s1_end = df.loc[df["source_sheet"] == "Year 2009-2010", "invoice_date"].max()
overlap = (df["source_sheet"] == "Year 2010-2011") & (df["invoice_date"] <= s1_end)
df2 = df[~overlap].copy()
print("overlap rows dropped:", overlap.sum())   # expect 22523
print("rows after overlap:", len(df2))          # expect 1044848

overlap rows dropped: 22523
rows after overlap: 1044848


In [3]:
EXCLUDE = ["POST", "C2", "DOT", "AMAZONFEE", "BANK CHARGES", "B", "ADJUST",
           "ADJUST2", "D", "M", "S", "CRUK", "TEST001"]

writeoff = ((df2["quantity"] < 0)
            & ~df2["invoice"].str.startswith("C")
            & df2["customer_id"].isna())
badprice = df2["price"] <= 0
nonprod = df2["stock_code"].isin(EXCLUDE) | df2["stock_code"].str.startswith("gift_")

print("write-offs:", writeoff.sum())      # expect 3393
print("price <= 0:", badprice.sum())      # expect 6029
print("non-product:", nonprod.sum())      # expect 5804

clean = df2[~(writeoff | badprice | nonprod)].copy()

write-offs: 3393
price <= 0: 6029
non-product: 5804


In [4]:
clean["customer_id"] = clean["customer_id"].astype("Int64")
clean["is_cancellation"] = clean["invoice"].str.startswith("C")
clean["has_customer"] = clean["customer_id"].notna()

print("rows:", len(clean))                       # expect 1033068
print("customers:", clean["customer_id"].nunique())   # expect 5875
print("net revenue:", round(clean["revenue"].sum()))   # expect about 18981278

rows: 1033068
customers: 5875
net revenue: 18981278


In [5]:
clean = df2[~(writeoff | badprice | nonprod)].copy()
clean["customer_id"] = clean["customer_id"].astype("Int64")
clean["is_cancellation"] = clean["invoice"].str.startswith("C")
clean["has_customer"] = clean["customer_id"].notna()

print("rows:", len(clean))                                   # expect 1,033,068
print("customers:", clean["customer_id"].nunique())
print("rows with customer id:", clean["has_customer"].mean().round(3))
print("revenue share with customer id:",
      (clean.loc[clean.has_customer, "revenue"].sum() / clean["revenue"].sum()).round(3))
print("cancellation rows:", clean["is_cancellation"].sum(),
      " revenue:", round(clean.loc[clean.is_cancellation, "revenue"].sum()))

show = ["invoice", "stock_code", "description", "quantity", "price", "customer_id", "revenue"]
print(clean.nlargest(5, "quantity")[show])
print(clean.nsmallest(5, "quantity")[show])
print(clean.nlargest(5, "revenue")[show])

clean.to_csv("../data/processed/online_retail_clean.csv", index=False)

rows: 1033068
customers: 5875
rows with customer id: 0.78
revenue share with customer id: 0.865
cancellation rows: 17974  revenue: -719693
        invoice stock_code                         description  quantity  \
1065882  581483      23843         PAPER CRAFT , LITTLE BIRDIE     80995   
587080   541431      23166      MEDIUM CERAMIC TOP STORAGE JAR     74215   
90857    497946      37410  BLACK AND WHITE PAISLEY FLOWER MUG     19152   
127166   501534      21099         SET/6 STRAWBERRY PAPER CUPS     12960   
127168   501534      21091         SET/6 WOODLAND PAPER PLATES     12960   

         price  customer_id   revenue  
1065882   2.08        16446  168469.6  
587080    1.04        12346   77183.6  
90857     0.10        13902    1915.2  
127166    0.10        13902    1296.0  
127168    0.10        13902    1296.0  
         invoice stock_code                          description  quantity  \
1065883  C581484      23843          PAPER CRAFT , LITTLE BIRDIE    -80995   
587085  

In [6]:
clean.to_csv("../data/processed/online_retail_clean.csv", index=False)